1.消息类型

在langchain中，发送给LLM的消息、LLM返回的消息都是统一被封装成BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们不需要自己创建BaseMessage对象，LangChain提供了一些常用的消息类型，如HumanMessage、SystemMessage、AIMessage。

In [16]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location."""
    return f"current weather in {location} is cloudy"

# 创建Agent
agent = create_agent(model = "deepseek-v4-pro", tools = [get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        # {"role": "system", "content": "你是一个热心的AI助手"},
        # {"role": "user", "content": "你好，哈哈哈我是强子"},
        # {"role": "assistant", "content": "你好，强子，很开心人认识你"},
        # {"role": "user", "content": "现在上海天气如何"},
        SystemMessage(content = "你是一个热心的AI助手"),
        HumanMessage(content = "你好，哈哈哈我是强子"),
        AIMessage(content = "你好，强子，很开心认识你"),
        HumanMessage(content = "现在上海天气如何")
    ]
})

print(response)

{'messages': [SystemMessage(content='你是一个热心的AI助手', additional_kwargs={}, response_metadata={}, id='2329a256-c96c-4222-9125-b7c181089afc'), HumanMessage(content='你好，哈哈哈我是强子', additional_kwargs={}, response_metadata={}, id='d981cc9c-e95f-4411-8181-763537ade956'), AIMessage(content='你好，强子，很开心认识你', additional_kwargs={}, response_metadata={}, id='1fcabc3c-910e-4827-80f6-a75aedfd6d7f', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='现在上海天气如何', additional_kwargs={}, response_metadata={}, id='4cb92ac1-a1f9-4618-b84e-1f40bb19454b'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': '用户想知道上海现在的天气。让我调用天气查询工具来获取上海的最新天气。\n\n用户提到了上海，我需要将"上海"作为location参数传递。'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 303, 'total_tokens': 379, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 31, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cac

In [17]:
for message in response["messages"]:
    print(message.pretty_print())    # 美观打印

================================ System Message ================================

你是一个热心的AI助手
None
================================ Human Message =================================

你好，哈哈哈我是强子
None
================================== Ai Message ==================================

你好，强子，很开心认识你
None
================================ Human Message =================================

现在上海天气如何
None
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_PuHCbSNs4gW6KwY460956113)
 Call ID: call_00_PuHCbSNs4gW6KwY460956113
  Args:
    location: 上海
None
================================= Tool Message =================================
Name: get_weather

current weather in 上海 is cloudy
None
================================== Ai Message ==================================

强子，现在上海的天气是**多云**☁️。出门的话可以不用带伞，但如果你想以防万一，带把伞也不碍事哈。有啥需要帮忙的随时叫我！
None


2.多模态消息

2.1 在线图片

In [68]:
from langchain.chat_models import init_chat_model
import os
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# 初始化模型
model = init_chat_model(
    model = "qwen3.7-plus",  # 多模态模型
    model_provider = "openai",
    base_url = "https://ws-3ssufixwbthuxno3.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

In [69]:
# 创建智能体Agent
agent = create_agent(model = model)

In [72]:
# 准备多模态消息
# message = {
#     "role": "user",
#     "content": [
#         {"role": "text", "text": "描述一下这个图片的内容"},
#         {"role": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg "},
#     ]
# }

message = HumanMessage(
    content = [
        {"type": "text", "text": "描述一下这个图片的内容"},
        {"type": "image_url", "image_url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [73]:
stream = agent.stream(
    {
    "messages": [message]
    },
    stream_mode = "messages"
)

for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end = "", flush = True)

这张图片描绘了一个非常温馨、和谐的海滩场景，看起来是在日落或日出时分（黄金时刻），因为光线非常柔和且带有温暖的金色调。以下是详细的画面内容描述：

1.  **主要人物与动物**：
    *   **年轻女子**：画面右侧坐着一位年轻女性，她留着长发，身穿黑白格子的长袖衬衫和深色长裤（裤脚卷起）。她面带灿烂的微笑，神情非常开心和放松。
    *   **狗狗**：画面左侧坐着一只大型犬，看起来像是一只黄色的拉布拉多寻回犬。它身上穿着带有彩色图案的深色胸背带。

2.  **互动动作**：
    *   两人正在进行亲密的互动。狗狗乖巧地坐着，抬起右前爪，搭在女子伸出的左手上，像是在“握手”或“击掌”。
    *   女子的右手似乎捏着一个小东西（很可能是零食），这暗示她可能正在训练狗狗，或者刚刚奖励了狗狗的乖巧表现。

3.  **环境与背景**：
    *   **沙滩**：他们坐在柔软的沙滩上，沙地上有一些脚印和纹理。在狗狗的身后，沙滩上放着一根红色的牵引绳。
    *   **大海**：背景是广阔的海面，可以看到海浪正在轻轻拍打着海岸。
    *   **光线**：阳光从画面的右侧（女子的身后）照射过来，形成了逆光效果，给女子的头发边缘和沙滩镀上了一层金色的光晕，营造出一种宁静、温暖且充满爱的氛围。

总的来说，这是一张展示人与宠物之间深厚感情和快乐时光的照片。

2.2本地图片数据

首先，我们得安装一个上传组件，用于模拟图片上传

In [ ]:
# uv add ipywidgets

In [84]:
from ipywidgets import FileUpload
from IPython.display import display
from langchain_core.messages import HumanMessage

uploader = FileUpload(accept = '*', multiple = False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [85]:
print(uploader.value)

({'name': '大雨海棠4.png', 'type': 'image/png', 'size': 584339, 'content': <memory at 0x000002A0BE147100>, 'last_modified': datetime.datetime(2024, 9, 1, 5, 18, 1, 33000, tzinfo=datetime.timezone.utc)},)


In [86]:
# 读取图片，转为 base64字符串
import base64

# 获取一个上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file['content']

# 转换内存视图 -> 字节
img_bytes = bytes(content_mv)

# base64 编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [87]:
# 组织多模态消息
multimodal_question = HumanMessage(content = [
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片的内容"}
])


for chunk, metadata in agent.stream(
        {"messages": [multimodal_question]},
        stream_mode = "messages"
):
    print(chunk.content, end = "", flush = True)

这是一张充满治愈感和温馨氛围的动画电影截图。从画风和人物特征来看，这出自日本动画电影**《海的迷途之家》**（The House of the Lost on the Cape）。

以下是画面的详细内容描述：

**1. 场景与环境：**
*   **时间与光影：** 画面描绘的是**日落时分**（黄昏）。夕阳悬挂在海平面上，散发出柔和的金光，将天空染成了温暖的橙黄色和淡淡的粉紫色。
*   **大海：** 广阔的海面波光粼粼，倒映着夕阳的光辉。远处海面上有一群飞鸟正在掠过，增加了画面的动感。
*   **前景：** 角色们站在一片长满青草的高地上，面前有一道简易的**木制栅栏**，将草地与大海隔开。

**2. 人物与动作：**
画面中有三个“角色”背对着镜头，面向大海，似乎在进行某种仪式或呼喊：
*   **中间的少女：** 一位扎着马尾辫的年轻女孩（这是电影中的主角之一“结”），身穿白色无袖上衣和深色七分裤，赤着脚。她双手拢在嘴边做成喇叭状，正在向大海大声呼喊。
*   **右边的小女孩：** 一个背着斗笠（看起来像传统的圆形草帽或背篓）的小女孩（这是电影中的“日和”），穿着短裤。她也模仿着少女的动作，双手拢在嘴边，一起向大海呼喊。
*   **左边的黑狗：** 一只黑白相间的小狗（看起来像柴犬），后腿站立，前爪搭在木栏杆上，也在专注地眺望着大海。

**3. 文字内容：**
*   画面底部有一个白色的字幕**“喂”**。这对应了人物向大海呼喊的动作，可能是在向远方的人打招呼，或者是在向大海倾诉心声。

**总结：**
这张图片传达了一种宁静、释放和充满希望的情感。三个角色（两个女孩和一只狗）共同面对壮丽的夕阳大海，象征着他们之间建立的羁绊以及面对新生活的勇气。